# Invoice PDF → Result
Select **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Upload one PDF when prompted. The final invoice JSON is displayed and downloaded automatically. Missing or uncertain fields remain marked for review. If the runtime reconnects and loses notebook variables, the upload cell detects or rebuilds the OCR setup automatically.


In [ ]:
#@title 1. Load project
import os
import pathlib
import subprocess

PROJECT_REF = 'main' #@param {type:"string"}
PROJECT_DIR = pathlib.Path('/content/OCR')
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', PROJECT_REF, '--depth', '1', 'https://github.com/ubaid-148/OCR.git', str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR / '.git').is_dir():
    current_branch = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'branch', '--show-current'], text=True).strip()
    if current_branch != PROJECT_REF:
        raise RuntimeError(f'Existing clone is on {current_branch}, expected {PROJECT_REF}. Start a fresh runtime to test the selected branch.')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    raise ValueError('/content/OCR exists but is not a Git clone. Start a fresh runtime.')
os.chdir(PROJECT_DIR)
print('Project ready at', PROJECT_DIR)
print('Testing commit:', subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
#@title 2. Prepare OCR
import pathlib
import sys

PROJECT_DIR = pathlib.Path('/content/OCR')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from colab_runtime import prepare_runtime

REQUIRE_GPU = False #@param {type:"boolean"}
OCR_PYTHON = prepare_runtime(PROJECT_DIR, require_gpu=REQUIRE_GPU)

EXTRACTION_MODE = "fast" #@param ["auto", "fast"]
if EXTRACTION_MODE == "auto":
    from colab_vision import prepare_vision_runtime
    prepare_vision_runtime()


In [ ]:
#@title 3. Upload PDF and get invoice result
import json
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path
from google.colab import files

DOWNLOAD_DIAGNOSTICS = False #@param {type:"boolean"}
OCR_LANGUAGES = 'eng+ara' #@param ['eng+ara', 'eng', 'ara', 'eng+urd']
PROJECT_DIR = Path(globals().get('PROJECT_DIR', '/content/OCR'))
if 'OCR_PYTHON' not in globals() or not Path(OCR_PYTHON).is_file():
    PROJECT_REF = globals().get('PROJECT_REF', 'main')
    runtime_helper = PROJECT_DIR / 'colab_runtime.py'
    if not runtime_helper.is_file():
        if not PROJECT_DIR.exists():
            subprocess.run(['git', 'clone', '--branch', PROJECT_REF, '--depth', '1', 'https://github.com/ubaid-148/OCR.git', str(PROJECT_DIR)], check=True)
        elif (PROJECT_DIR / '.git').is_dir():
            subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
        else:
            raise RuntimeError('/content/OCR exists but is not a Git clone. Start a fresh runtime.')
    if not runtime_helper.is_file():
        raise RuntimeError('OCR setup helper is missing. Start a fresh runtime and run this cell again.')
    if str(PROJECT_DIR) not in sys.path:
        sys.path.insert(0, str(PROJECT_DIR))
    from colab_runtime import prepare_runtime
    print('OCR setup was not active; preparing it now.', flush=True)
    OCR_PYTHON = prepare_runtime(PROJECT_DIR, require_gpu=False)
EXTRACTION_MODE = globals().get("EXTRACTION_MODE", "fast")
if EXTRACTION_MODE == "auto":
    from colab_vision import prepare_vision_runtime
    prepare_vision_runtime()
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one PDF.')
original_name, pdf_bytes = next(iter(uploaded.items()))
if Path(original_name).suffix.lower() != '.pdf' or not pdf_bytes.startswith(b'%PDF-'):
    raise ValueError('Please upload a valid PDF.')

with tempfile.TemporaryDirectory(prefix='invoice-result-') as temporary:
    run_dir = Path(temporary)
    pdf_path = run_dir / 'input.pdf'
    raw_path = run_dir / 'raw_ocr.json'
    clean_path = run_dir / 'invoice.json'
    details_path = run_dir / 'extraction_details.json'
    pdf_path.write_bytes(pdf_bytes)
    commands = [
        [OCR_PYTHON, '-u', str(PROJECT_DIR / 'coordinate_ocr.py'),
         str(pdf_path), str(raw_path), OCR_LANGUAGES],
        [OCR_PYTHON, str(PROJECT_DIR / 'invoice_result.py'), str(raw_path),
         '--output', str(clean_path), '--filename', original_name, '--language', OCR_LANGUAGES,
         '--mode', EXTRACTION_MODE, '--pdf', str(pdf_path), '--details-output', str(details_path)],
    ]
    print('Reading invoice…')
    for command in commands:
        process = subprocess.run(command, cwd=PROJECT_DIR, env=os.environ.copy(),
                                 capture_output=True, text=True)
        if process.returncode:
            raise RuntimeError(process.stderr[-12000:] or process.stdout[-12000:] or 'Invoice extraction failed')
    invoice = json.loads(clean_path.read_text(encoding='utf-8'))
    raw_summary = json.loads(raw_path.read_text(encoding='utf-8'))
    extraction_details = json.loads(details_path.read_text(encoding='utf-8'))
    raw_summary['extraction_details'] = extraction_details

results_dir = PROJECT_DIR / 'benchmark_outputs' / 'invoices'
results_dir.mkdir(parents=True, exist_ok=True)
filename = re.sub(r'[^A-Za-z0-9_.-]', '_', Path(original_name).stem) + '-invoice.json'
invoice_path = results_dir / filename
invoice_path.write_text(json.dumps(invoice, ensure_ascii=False, indent=2), encoding='utf-8')
print('Extraction mode:', EXTRACTION_MODE)
print('Parser:', extraction_details.get('quality', {}).get('parser', 'unknown'))
print('AI status:', extraction_details.get('quality', {}).get('local_ai_status', 'not_used'))
print('Extraction timings:', extraction_details.get('stage_timings', {}))
print('Vision diagnostics:', json.dumps(extraction_details.get('vision_diagnostics', []), ensure_ascii=False))
print('OCR device:', raw_summary.get('device', 'unknown'))
print('Pipeline version:', raw_summary.get('pipeline_version', 'unknown'))
print('OCR timings (seconds):', json.dumps(raw_summary.get('timings_seconds', {})))
print(json.dumps(invoice, ensure_ascii=False, indent=2))
diagnostic_path = results_dir / (Path(filename).stem + '-diagnostics.json')
diagnostic_path.write_text(json.dumps(raw_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('OCR diagnostics saved:', diagnostic_path)
files.download(str(invoice_path))
if DOWNLOAD_DIAGNOSTICS:
    files.download(str(diagnostic_path))
